# Code to get started

In [10]:
import pandas as pd
import numpy as np

In [11]:
# 1. Load the gold data
df = pd.read_csv('gold_housing_data.csv')

In [12]:
# 2. Separate numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols 

['Id',
 'MSSubClass',
 'LotFrontage',
 'LotArea',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtFinSF1',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'MiscVal',
 'MoSold',
 'YrSold',
 'SalePrice']

In [13]:
# Remove the target variable and ID from the feature list
if 'SalePrice' in numerical_cols:
    numerical_cols.remove('SalePrice')
if 'Id' in numerical_cols:
    numerical_cols.remove('Id')

print(f"Numerical columns to process: {numerical_cols}")

Numerical columns to process: ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']


In [14]:
# 3. Check for Skewness (values > 0.75 are usually candidates for Log Transformation)
skewed_features = df[numerical_cols].skew().sort_values(ascending=False)
print("\nSkewness of numerical features:")
print(skewed_features[abs(skewed_features) > 0.75])


Skewness of numerical features:
MiscVal          24.476794
PoolArea         14.828374
LotArea          12.207688
3SsnPorch        10.304342
LowQualFinSF      9.011341
KitchenAbvGr      4.488397
BsmtFinSF2        4.255261
ScreenPorch       4.122214
BsmtHalfBath      4.103403
EnclosedPorch     3.089872
MasVnrArea        2.677616
LotFrontage       2.409147
OpenPorchSF       2.364342
BsmtFinSF1        1.685503
WoodDeckSF        1.541376
TotalBsmtSF       1.524255
MSSubClass        1.407657
1stFlrSF          1.376757
GrLivArea         1.366560
BsmtUnfSF         0.920268
2ndFlrSF          0.813030
dtype: float64


# Log Transformation

In [15]:
# 1. Get the list of highly skewed features from your previous result
skewed_cols = skewed_features[abs(skewed_features) > 0.75].index

In [16]:
# 2. Apply the log transformation
for col in skewed_cols:
    df[col] = np.log1p(df[col])

In [17]:
# 3. Verify that the skewness has decreased
print("New Skewness values (should be much lower):")
print(df[skewed_cols].skew().sort_values(ascending=False))

New Skewness values (should be much lower):
PoolArea         14.363102
3SsnPorch         7.734975
LowQualFinSF      7.460317
MiscVal           5.170704
BsmtHalfBath      3.933064
KitchenAbvGr      3.869414
ScreenPorch       3.150409
BsmtFinSF2        2.523694
EnclosedPorch     2.112275
MasVnrArea        0.503532
2ndFlrSF          0.289643
MSSubClass        0.248997
WoodDeckSF        0.153537
1stFlrSF          0.080114
GrLivArea        -0.006140
OpenPorchSF      -0.023397
LotArea          -0.137404
BsmtFinSF1       -0.618410
LotFrontage      -0.870006
BsmtUnfSF        -2.186504
TotalBsmtSF      -5.154670
dtype: float64


#### This is a great result! We can see that features like GrLivArea, 2ndFlrSF, and MSSubClass are now almost perfectly balanced (skewness near zero), which is exactly what we wanted.
### A Quick Note on the "Negative" Skewness
#### We might notice some values, like TotalBsmtSF or BsmtUnfSF, are now highly negative (e.g., -5.15).
#### Why this happened: The np.log1p transformation was likely "too strong" for these specific columns, over-correcting them in the other direction.
#### Should we worry? Not at all. For a Random Forest model, this does not hurt performance. Tree-based models are "scale-invariant," meaning they don't care if the data is perfectly normal or slightly skewed. The fact that you have reduced the extreme, erratic outliers (like the original MiscVal at 24+) is the most important achievement for your model’s stability.

# The Final Step: Export

In [18]:
# Save your numerically engineered data
df.to_csv('numerical_engineered_data.csv', index=False)
print("Numerical features engineered and saved!")

Numerical features engineered and saved!
